# Formatting Anki Flashcards

## Goal

While using Anki over the years, I struggled with following some common best practices―like keeping each card atomic, choosing the best direction to test, avoiding topic-labels, having clear cues, etc. Building great cards takes time which, in my experience, it's a major friction to creating new cards and reviewing them. 

Throughout this notebook, we want to iteratively build a good way to prompt a small language model, following basic best practices. We will start with a simple LLM call and systematically build a more robust and performant solution. 

We will follow this progression:
1. Simple prompt
1. Simple prompt + Structured Output
1. Simple prompt + Structured Output + Thinking
1. Improve prompt based on error analysis and frequent failure modes

## Setup environment

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from anki.collection import Collection

from addon.application.use_cases.note_counter import is_note_marked_for_review
from addon.infrastructure.configuration.settings import AddonConfig

In [3]:
# Open an existing collection
col = Collection("/home/gianluca/.local/share/Anki2/User 1/collection.anki2")

# Do something with the collection
print(f"Number of notes: {col.note_count()}")
print(f"Number of cards: {col.card_count()}")

Number of notes: 3195
Number of cards: 3257


### Connect to Inference Server

In [4]:
from IPython.display import Markdown, display

from addon.domain.entities.note import AddonNote
from addon.infrastructure.llm.schemas import AddonNoteChanges


def display_addon_note(addon_note: AddonNote | AddonNoteChanges | str) -> None:
    if isinstance(addon_note, str):
        print(addon_note)
    elif isinstance(addon_note, (AddonNote, AddonNoteChanges)):
        display(
            Markdown(
                f"**Front:** {addon_note.front}\n\n"
                f"**Back:** {addon_note.back}\n\n"
                # f"**Tags:** {addon_note.tags}"
            )
        )
    else:
        raise ValueError(
            f"Expected str, AddonNote, or AddonNoteChanges, found {type(addon_note)}"
        )

In [5]:
from addon.infrastructure.external_services.openai import OpenAIClient


def answer(input: str | list[dict], guided_json=None, **kwargs):
    """Helper function to prompt LLM.

    input: Either a string (completions API) or list of message dicts (chat completion)
    kwargs: Can override config values like max_tokens, temperature, etc.
            extra_body: dict whose contents are merged into the request payload.
    """
    extra_body = kwargs.pop("extra_body", {})
    config = AddonConfig.create_nullable(kwargs)
    client = OpenAIClient.create(config)

    run_kwargs = {**extra_body}
    if guided_json is not None:
        run_kwargs["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": "addon_note_changes",
                "schema": guided_json,
            },
        }

    content = client.run(input, **run_kwargs)
    reasoning_content = client.last_reasoning_content
    cleaned_content = content.replace("<think>\n\n</think>\n\n", "")

    if reasoning_content:
        print("=== Thinking ===")
        print(reasoning_content)
        print("=== End Thinking ===\n")

    display_addon_note(cleaned_content)

    if guided_json is not None:
        suggested_changes = AddonNoteChanges.model_validate_json(
            cleaned_content
        )
        return (suggested_changes, reasoning_content)

    return (cleaned_content, reasoning_content)

Now we have everything we need to tell the LLM to make some changes to our Anki flashcards.

Let's pull a few notes currently marked for review.

In [6]:
deck_id = col.decks.current()["id"]
query = f"did:{deck_id}"
note_ids = col.find_notes(query)

NUM_NOTES_NEEDED = 3

flagged_notes = []
for note_id in note_ids:
    if is_note_marked_for_review(col, note_id):
        note = col.get_note(note_id)
        if "personal" not in note.tags:
            flagged_notes.append(note)
            if len(flagged_notes) >= NUM_NOTES_NEEDED:
                break

In [7]:
print(f"Number of flagged notes: {len(flagged_notes)}")

Number of flagged notes: 3


## Show original version of flagged notes

In [8]:
from addon.application.services.formatter_service import AnkiNoteAdapter

addon_notes = [AnkiNoteAdapter.to_addon_note(note) for note in flagged_notes]

for i, note in enumerate(addon_notes):
    print(f"--- Note {i + 1} ---")
    display_addon_note(note)
    print()

--- Note 1 ---


**Front:** How can we compute the length of a vector $\overrightarrow{v} = \begin{bmatrix} v_{1} \\ v_{2} \end{bmatrix}$?

**Back:** $\left\|\mathbf{v}\right\| = \sqrt{v_1^2 + v_2^2}$




--- Note 2 ---


**Front:** L2-norm: also known as

**Back:** Euclidian distance




--- Note 3 ---


**Front:** Name for _bagging_ in Statistics

**Back:** Bootstrapping



## Simple Prompt

Let's build a simple prompt, instructing the LLM to improve Anki flashcards based on some basic guidelines.

In [9]:
simple_prompt_template = (
    "Look at this flashcard. How would you improve it? "
    "Keep in mind that flashcards should be atomic, concise, and accurate. "
    "Do not explain the changes.\n\nFront: {note.front}\nBack: {note.back}"
)

# Print a sample prompt
print(simple_prompt_template.format(note=addon_notes[0]))

Look at this flashcard. How would you improve it? Keep in mind that flashcards should be atomic, concise, and accurate. Do not explain the changes.

Front: How can we compute the length of a vector $\overrightarrow{v} = \begin{bmatrix} v_{1} \\ v_{2} \end{bmatrix}$?
Back: $\left\|\mathbf{v}\right\| = \sqrt{v_1^2 + v_2^2}$


In [10]:
for i, addon_note in enumerate(addon_notes):
    print(f"--- Note {i + 1} ---")
    answer(
        input=[
            {
                "role": "user",
                "content": simple_prompt_template.format(note=addon_note),
            }
        ],
        max_tokens=1_000,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    print()

--- Note 1 ---
Front: What is the length of vector $\mathbf{v} = \begin{bmatrix} v_1 \\ v_2 \end{bmatrix}$?
Back: $\|\mathbf{v}\| = \sqrt{v_1^2 + v_2^2}$

--- Note 2 ---
Front: The L2-norm is also known as
Back: Euclidean distance

--- Note 3 ---
Front: What is the statistical term for resampling with replacement?
Back: Bootstrapping



Despite the very simple prompt, on this flashcard, the model is doing a good job. However, we probably need some structured output/constrained decoding to ensure the LLM respects a predefined format. This will make extracting the suggestions more easier.

## Simple Prompt + Structured Output

Let's reuse the `pydantic` schema we have already defined for the main project.

In [11]:
from addon.infrastructure.llm.schemas import AddonNoteChanges

for i, addon_note in enumerate(addon_notes):
    print(f"--- Note {i + 1} ---")
    content, _ = answer(
        input=[
            {
                "role": "user",
                "content": simple_prompt_template.format(note=addon_note),
            }
        ],
        guided_json=AddonNoteChanges.model_json_schema(),
        max_tokens=2_000,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    display_addon_note(content)
    print()

--- Note 1 ---
{
  "front": "Formula for the length of a 2D vector $\\begin{bmatrix} v_1 \\\\ v_2 \\end{bmatrix}$?",
  "back": "$\\sqrt{v_1^2 + v_2^2}$"
}


**Front:** Formula for the length of a 2D vector $\begin{bmatrix} v_1 \\ v_2 \end{bmatrix}$?

**Back:** $\sqrt{v_1^2 + v_2^2}$




--- Note 2 ---
{
  "front": "L2-norm (Euclidean norm) formula for a vector $\\mathbf{x}$?",
  "back": "$\\sqrt{\\sum x_i^2}$"
}


**Front:** L2-norm (Euclidean norm) formula for a vector $\mathbf{x}$?

**Back:** $\sqrt{\sum x_i^2}$




--- Note 3 ---
{
  "front": "What is the statistical term for resampling with replacement?",
  "back": "Bootstrapping"
}


**Front:** What is the statistical term for resampling with replacement?

**Back:** Bootstrapping



## Reasoning + Structured Output + Old Prompt

Let's compare it with the current version we have in the `main` branch.

In [12]:
from addon.application.services.formatter_service import NoteFormatter

config = AddonConfig.create_nullable(
    {
        "mode": "v1/chat/completions",
        "url": "http://iamgianluca.ddns.net:8080/v1/completions",
        "max_tokens": 15_000,  # needs headroom for reasoning + structured output
    }
)
for k, v in config.__dict__.items():
    if k == "url":  # hide inference server url
        continue
    print(f"{k}: {v}")

model_name: ./Qwen3.5-27B-UD-Q4_K_XL.gguf
temperature: 1.0
max_tokens: 15000
top_p: 0.95
top_k: 20
min_p: 0.0


In [13]:
openai = OpenAIClient.create(config)
formatter = NoteFormatter(openai)

for i, addon_note in enumerate(addon_notes):
    print(f"--- Note {i + 1} ---")
    new_note = formatter.format(note=addon_note)
    display_addon_note(new_note)
    print()

--- Note 1 ---


**Front:** Length of vector $\overrightarrow{v} = \begin{bmatrix} v_{1} \\ v_{2} \end{bmatrix}$

**Back:** $\left\|\mathbf{v}\right\| = \sqrt{v_1^2 + v_2^2}$, 




--- Note 2 ---


**Front:** L2-norm alias

**Back:** Euclidean distance

$ \|x\|_2 = \sqrt{\sum_{i=1}^{n} x_i^2} $ (Euclidean norm formula)

Note: Represents Euclidean distance from origin to point x in n-dimensional space. Also called Euclidean norm or 2-norm in linear algebra and machine learning contexts.

$ d(x, y) = \|x - y\|_2 = \sqrt{\sum_{i=1}^{n} (x_i - y_i)^2} $ (Euclidean distance between points x and y)

Used in:
* Loss functions (MSE regression)
* Regularization (Ridge regression)
* Neural networks (weight constraints)
* k-NN algorithms
* Clustering (K-means initialization)
* Physics (displacement magnitude)

Contrast with:
* L1-norm: $ \|x\|_1 = \sum |x_i| $ (Manhattan distance)
* L∞-norm: $ \|x\|_\infty = \max |x_i| $ (Chebyshev distance)
* Frobenius norm: $ \|A\|_F = \sqrt{\sum_{i,j} |a_{ij}|^2} $ (matrix L2-norm)

Properties:
* Scale invariant under orthogonal transformations
* Differentiable everywhere except at origin
* Induces standard topology in $\mathbb{R}^n$
* Used in gradient descent due to smoothness

Historical note: Named after mathematician Augustin-Louis Cauchy who formalized norm concepts in 19th century. Common misconception: not to be confused with squared Euclidean distance ($ \sum x_i^2 $) used in some optimization algorithms.

Common applications:
* Computer vision (image similarity metrics)
* Recommender systems (collaborative filtering)
* Natural language processing (word embeddings)
* Robotics (path planning)
* Signal processing (energy calculations)

Key insight: The square of L2-norm is often used in optimization ($ \|x\|^2 $) to avoid square root computation while preserving minimization properties. This leads to Ridge regression formulation: $ \text{min} \sum(y_i - f(x_i))^2 + \lambda\|w\|_2^2 $.

Important caveat: In high-dimensional spaces, L2-norm can suffer from curse of dimensionality where distances between points become increasingly similar. This affects clustering and nearest-neighbor algorithms requiring careful preprocessing.

Related concepts: Cosine similarity (angle-based), Mahalanobis distance (correlation-aware), Minkowski distance (generalized p-norm). L2-norm is specific case of Minkowski with p=2.

Computational complexity: O(n) for single vector norm calculation, O(n²) for matrix Frobenius norm. Optimized BLAS/LAPACK implementations exist for efficient computation in scientific computing.

Numerical stability: For very large/small values, direct computation may cause overflow/underflow. Log-space transformations or careful scaling recommended in practical implementations.

Geometric interpretation: Unit circle in 2D L2-norm space. Unit ball becomes hypersphere in n-dimensions. This rotational symmetry property distinguishes L2 from other norms.

Statistical connection: L2-norm relates to variance and standard deviation. Sample variance formula uses squared deviations (L2 concept) from mean. This connects probabilistic interpretations with distance metrics.

Machine learning impact: L2 regularization (weight decay) adds $ \lambda\|w\|_2^2 $ to loss function. Encourages smaller weights distributed across features rather than sparse solutions like L1 regularization. Affects model complexity control.

Deep learning specifics: In batch normalization, L2-norm concepts appear in standardization formulas. Layer normalization applies similar concepts across feature dimensions. Critical for training stability in deep networks.

Optimization landscape: Gradient of L2-norm is $ \frac{\partial}{\partial x} \|x\|_2 = \frac{x}{\|x\|_2} $. At origin gradient undefined - special handling required in optimization algorithms. This non-differentiability at zero is key difference from L1.

Real-world example: GPS navigation systems calculate straight-line distances using Euclidean (L2) metric. While actual travel paths follow roads (graph metrics), initial route planning often uses L2 approximation for efficiency.

Common pitfalls: Confusing L2-norm of difference ($ \|x-y\|_2 $) with squared Euclidean distance. Also mixing up vector norms with matrix norms. Always specify dimensionality and context when discussing L2 properties.

Implementation tip: In NumPy/Python: `np.linalg.norm(x, 2)` computes L2-norm. For efficiency, `np.sqrt(np.sum(x**2))` avoids function call overhead in performance-critical loops. PyTorch equivalent: `torch.norm(x, p=2)`.

Advanced note: In functional analysis, L2-space refers to Hilbert space of square-integrable functions. This abstract concept generalizes finite-dimensional L2-norm to infinite dimensions - foundation of quantum mechanics and signal theory.

Final reminder: While L2-norm dominates many applications, always verify if alternative metrics (L1, Mahalanobis, etc.) better suit your specific problem domain and data characteristics.

Tags: ['math'] }




--- Note 3 ---


**Front:** Statistical name for _bagging_

**Back:** Bootstrapping

```bash
$ bootstrap <sample>
```

`bagging` = bootstrap aggregating

`<S-p>` to copy

$ [\hat{y}_{bagging}] = \frac{1}{T} \sum_{t=1}^{T} \hat{y}_t $", 



The current version in production, only barely change to the original flashcard, and focuses mostly on formatting. This is likely due to the highly specific prompt. A shorter and more generic prompt, in combination with reasoning, could lead to better results.

For reference, below is the original note.

## Reasoning + Structured Output + Simple Prompt

Let's try to let the LLM reason, while still perform constrained decoding.

In [14]:
from addon.infrastructure.llm.schemas import AddonNoteChanges

for i, addon_note in enumerate(addon_notes):
    print(f"--- Note {i + 1} ---")
    content, _ = answer(
        input=[
            {
                "role": "user",
                "content": simple_prompt_template.format(note=addon_note),
            }
        ],
        guided_json=AddonNoteChanges.model_json_schema(),
        max_tokens=1_000,
        extra_body={"chat_template_kwargs": {"enable_thinking": True}},
    )
    display_addon_note(content)
    print()

--- Note 1 ---
{
  "front": "Formula for the Euclidean norm of a 2D vector $\\mathbf{v} = \\begin{bmatrix} v_1 \\\\ v_2 \\end{bmatrix}$",
  "back": "$\\|\\mathbf{v}\\| = \\sqrt{v_1^2 + v_2^2}$"
}


**Front:** Formula for the Euclidean norm of a 2D vector $\mathbf{v} = \begin{bmatrix} v_1 \\ v_2 \end{bmatrix}$

**Back:** $\|\mathbf{v}\| = \sqrt{v_1^2 + v_2^2}$




--- Note 2 ---
{
  "front": "Alternative name for the L2-norm",
  "back": "Euclidean distance"
}


**Front:** Alternative name for the L2-norm

**Back:** Euclidean distance




--- Note 3 ---
{
  "front": "What is another name for _bagging_ in Statistics?",
  "back": "Bootstrapping (or Bootstrap aggregating)"
}


**Front:** What is another name for _bagging_ in Statistics?

**Back:** Bootstrapping (or Bootstrap aggregating)



In [15]:
col.close()

## TODO

- [x] Simple prompt
- [x] Simple prompt + Constrained Decoding
- [x] Simple prompt + Constrained Decoding + Reasoning
- [ ] Simple prompt + Chain of Thought
- [ ] Cleaner representation of existing addon note in the prompt passed to the LLM
- [ ] Pass note to change as `user` instead of `system` role
- [ ] Two-step process (critique → refine)
- [ ] Agent
- [ ] Check if we have a class that we can use to read the collection from the hard disk and convert it to `AddonNote` instead of `Note`. In that way we can operate with domain objects instead of external dependencies. The same class should be used to convert the `AddonNote` back to `Note` (so maybe we should keep track of the `note_id`
- [ ] Once we have that class, we should update the rest of the codebase accordingly (e.g., note formatter, note counter, etc.)
- [ ] ...